# Week 1 Day 2 — Tensor ops: reshape, matmul, broadcasting
**Jun 30, 2026**

## Part 1: reshape / view / squeeze / unsqueeze

In [ ]:
import torch

x = torch.arange(12, dtype=torch.float32)
print('Original:', x.shape)         # [12]

a = x.view(3, 4)
print('view(3,4):', a.shape)        # [3, 4]

b = x.reshape(2, 2, 3)
print('reshape(2,2,3):', b.shape)   # [2, 2, 3]

c = a.unsqueeze(0)                  # add dim at front
print('unsqueeze(0):', c.shape)     # [1, 3, 4]

d = c.squeeze(0)                    # remove dim
print('squeeze(0):', d.shape)       # [3, 4]

**`view` vs `reshape`:** `view` requires contiguous memory; `reshape` works even if not. Safe habit: prefer `reshape` unless you have a reason to use `view`.

In [ ]:
# Try yourself: reshape x into (6,2), then (12,1), then back to (12,)


## Part 2: matmul

In [ ]:
# 2D matrix multiply
A = torch.randn(3, 4)
B = torch.randn(4, 5)

C1 = torch.matmul(A, B)    # preferred
C2 = A @ B                 # same thing, cleaner
print('A @ B:', C1.shape)  # [3, 5]

# dot product (1D)
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, 5.0, 6.0])
print('dot product:', torch.dot(u, v))   # 32.0

In [ ]:
# Batched matmul — critical for attention
Q = torch.randn(8, 10, 64)         # batch=8, seq_len=10, d_k=64
K = torch.randn(8, 10, 64)
scores = Q @ K.transpose(-2, -1)   # [8, 10, 10] — attention score matrix
print('attention scores:', scores.shape)

In [ ]:
# Try yourself: compute softmax of scores along last dim
# attn_weights = torch.softmax(scores, dim=-1)
# print(attn_weights.shape)  — what do you get?


## Part 3: Broadcasting

Rule: dimensions are compared **right-to-left**. Each dim must be equal, or one of them must be 1 (it gets expanded).

In [ ]:
M = torch.ones(3, 4)

# scalar
print('M + 5:', (M + 5).shape)                                       # [3, 4]

# row vector
row = torch.tensor([1.0, 2.0, 3.0, 4.0])  # [4]
print('M + row:', (M + row).shape)                                   # [3, 4]

# col vector
col = torch.tensor([[1.0], [2.0], [3.0]])  # [3, 1]
print('M + col:', (M + col).shape)                                   # [3, 4]

# two broadcastable tensors
p = torch.randn(5, 1, 4)
q = torch.randn(1, 3, 4)
print('p + q:', (p + q).shape)                                       # [5, 3, 4]

In [ ]:
# Real use case: subtract mean per feature (feature normalization)
data = torch.randn(100, 64)    # 100 samples, 64 features
mean = data.mean(dim=0)        # [64]
centered = data - mean         # [100, 64] — broadcasts over samples
print('centered:', centered.shape)

In [ ]:
# Outer product via broadcasting
a = torch.tensor([1.0, 2.0, 3.0]).unsqueeze(1)   # [3, 1]
b = torch.tensor([4.0, 5.0, 6.0]).unsqueeze(0)   # [1, 3]
outer = a * b                                      # [3, 3]
print('outer product:\n', outer)

In [ ]:
# Common mistake — try this and read the error
x1 = torch.randn(3, 4)
x2 = torch.randn(4, 3)
# x1 + x2   # RuntimeError!
# fix:
print('fixed:', (x1 + x2.T).shape)

## Part 4: Useful utility ops

In [ ]:
t = torch.randn(4, 5)

print('shape:', t.shape)
print('dtype:', t.dtype)
print('device:', t.device)
print('transpose:', t.T.shape)               # [5, 4]
print('sum dim=0:', t.sum(dim=0).shape)      # [5]
print('sum dim=1:', t.sum(dim=1).shape)      # [4]
print('mean:', t.mean())
print('argmax:', t.argmax())

# type conversion
x_int = torch.tensor([1, 2, 3])
x_float = x_int.float()
print('float dtype:', x_float.dtype)

# GPU
if torch.cuda.is_available():
    t_gpu = t.cuda()
    print('on GPU:', t_gpu.device)
else:
    print('CPU only — fine for now')

## Summary

| Op | What it does |
|---|---|
| `reshape` / `view` | change shape without changing data |
| `unsqueeze(i)` / `squeeze(i)` | add / remove a dimension |
| `@` / `torch.matmul` | matrix multiply (works batched too) |
| `*` | element-wise multiply (NOT matmul) |
| broadcasting | right-align shapes; dims of 1 expand automatically |
| `transpose(-2,-1)` | swap last two dims — standard in attention code |